In [ ]:
FORCE_REINSTALL = False  # set True to reinstall even if the package is already installed

try:
    if FORCE_REINSTALL:
        raise ImportError("FORCE_REINSTALL is set")
    from mko_data_cleaner import __version__, check_notebook_environment
except ImportError:
    # Package missing, too old, or forced reinstall.
    # --force-reinstall clears the old version; --no-deps avoids
    # re-resolving the dependency tree (already present in the env).
    get_ipython().run_line_magic(
        "pip",
        'install --force-reinstall --no-deps "git+https://github.com/manicko/mko_data_cleaner.git@v0.2.0"',
    )  # noqa: E501
    from mko_data_cleaner import __version__, check_notebook_environment

print("mko_data_cleaner", __version__)
check_notebook_environment()


## Импорт зависимостей

In [ ]:
from pathlib import Path

import polars as pl
from mko_data_cleaner import process_data, ReportResult
import mko_data_cleaner

print("mko_data_cleaner", mko_data_cleaner.__version__)
print("process_data ->", process_data.__doc__.strip().splitlines()[0])

## Настройка путей и файлов отчета

In [ ]:
# Windows example (comment only):
# import os
# WORKSPACE_ROOT = Path(os.environ["LOCALAPPDATA"]) / "mko_get_mediascope_data" / "mko_get_mediascope_data" / "settings" / "connections" / "mediascope.json"
# WORKSPACE_ROOT = Path(platformdirs.user_config_dir(APP_NAME)) / "settings" /
# Example override to user config/external path (Windows example, comment only):
# WORKSPACE_ROOT = Path(r"%LOCALAPPDATA%\mko_get_mediascope_data\mko_get_mediascope_data\")

# Portable workspace root (override as needed).
WORKSPACE_ROOT = Path.cwd()

SETTINGS_DIR = WORKSPACE_ROOT / "02-settings"
REPORT_DIR = WORKSPACE_ROOT / "output"

## Expected report directory layout

```
report_dir/
  raw_data/<*.gz>         # gzipped data CSVs (extension from config)
  dict/merged_dictionary.csv   # plain merged dictionary (read directly)
  clean_data/             # created at run time: data_table_*.csv.gz
  data_base/db_example.db # created at run time (SQLite)
  null_data_*.csv.gz      # created at run time: rows with no dictionary match
```

## Run the pipeline

In [ ]:
result: ReportResult = process_data(REPORT_DIR, settings_dir=SETTINGS_DIR)
print(result)

## Inspect the result

`ReportResult` exposes `rows_loaded`, `rows_exported`, `timings` (`{import, mapping, export}`), `export_dir`, `null_data_files`, `import_dir`, `report_dir`, `db_path`, plus timing/duration fields.

In [ ]:
print("rows_loaded:", result.rows_loaded)
print("rows_exported:", result.rows_exported)
print("timings:", result.timings)
print("export_dir:", result.export_dir)
print("null_data_files:", result.null_data_files)
clean = next(result.export_dir.glob("*.csv.gz"))
print("clean_data sample:"); display(pl.read_csv(clean, separator=";").head())
if result.null_data_files:
    nd = result.null_data_files[0]
    print("null_data sample:"); display(pl.read_csv(nd, separator=";").head())